Week 11 Day 1

In [1]:
!pip install -q mlflow scikit-learn pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0

In [2]:
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Load dataset
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="target")

print("Dataset shape:", X.shape)
print("Classes:", iris.target_names)


Dataset shape: (150, 4)
Classes: ['setosa' 'versicolor' 'virginica']


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 120
Testing samples: 30


In [4]:
# Create/select an MLflow experiment
mlflow.set_experiment("W11_Iris_Classification")

print("Experiment created/selected successfully.")


2026/09/06 09:55:48 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/06 09:55:48 INFO mlflow.store.db.utils: Updating database tables
2026/09/06 09:55:53 INFO mlflow.tracking.fluent: Experiment with name 'W11_Iris_Classification' does not exist. Creating a new experiment.


Experiment created/selected successfully.


In [5]:
with mlflow.start_run(run_name="random_forest_v1"):

    # Model parameters
    n_estimators = 100
    max_depth = 5
    random_state = 42

    # Create model
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state
    )

    # Train
    model.fit(X_train, y_train)

    # Predictions
    predictions = model.predict(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(
        y_test,
        predictions,
        average="weighted"
    )
    recall = recall_score(
        y_test,
        predictions,
        average="weighted"
    )

    # Log parameters
    mlflow.log_param("model", "RandomForestClassifier")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", random_state)

    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)

    # Log model
    mlflow.sklearn.log_model(
        model,
        "model"
    )

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)

    print("\nMLflow run ID:", mlflow.active_run().info.run_id)


2026/09/06 09:55:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy : 0.9333333333333333
Precision: 0.9333333333333333
Recall   : 0.9333333333333333

MLflow run ID: 6c134eef0bf440cab32a18a72244f438


In [6]:
!nohup mlflow ui --host 0.0.0.0 --port 5000 > mlflow.log 2>&1 &


In [7]:
!pip install -q pyngrok


In [10]:
import mlflow

experiment = mlflow.get_experiment_by_name(
    "W11_Iris_Classification"
)

print("Experiment ID:", experiment.experiment_id)


Experiment ID: 1


In [11]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

runs[
    ["run_id", "status", "metrics.accuracy"]
]


,run_id,status,metrics.accuracy
0,6c134eef0bf440cab32a18a72244f438,FINISHED,0.933333


In [13]:
import mlflow

experiment = mlflow.get_experiment_by_name(
    "W11_Iris_Classification"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

print("Number of runs:", len(runs))
print("\nTracked experiments:")

display(
    runs[
        [
            "run_id",
            "status",
            "params.model",
            "params.n_estimators",
            "params.max_depth",
            "metrics.accuracy",
            "metrics.precision",
            "metrics.recall"
        ]
    ]
)


Number of runs: 1

Tracked experiments:


,run_id,status,params.model,params.n_estimators,params.max_depth,metrics.accuracy,metrics.precision,metrics.recall
0,6c134eef0bf440cab32a18a72244f438,FINISHED,RandomForestClassifier,100,5,0.933333,0.933333,0.933333


Day 2

In [14]:
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


In [15]:
mlflow.set_experiment("W11_Iris_Classification")

results = []

configs = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 200, "max_depth": 10},
]

for config in configs:

    with mlflow.start_run(
        run_name=f"rf_{config['n_estimators']}_{config['max_depth']}"
    ):

        model = RandomForestClassifier(
            n_estimators=config["n_estimators"],
            max_depth=config["max_depth"],
            random_state=42
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        accuracy = accuracy_score(
            y_test,
            predictions
        )

        mlflow.log_params(config)
        mlflow.log_metric("accuracy", accuracy)

        mlflow.sklearn.log_model(
            model,
            "model"
        )

        run_id = mlflow.active_run().info.run_id

        results.append({
            "run_id": run_id,
            "n_estimators": config["n_estimators"],
            "max_depth": config["max_depth"],
            "accuracy": accuracy
        })

results_df = pd.DataFrame(results)

results_df


2026/09/06 09:59:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/06 09:59:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/06 10:00:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


,run_id,n_estimators,max_depth,accuracy
0,8b8aaa3837204aea93a7f863d435fc4e,50,3,0.966667
1,322ea7cb0fc84b48884cfb2e85081328,100,5,0.933333
2,de8adde8c7b048e89d388a9efb75564c,200,10,0.900000


In [16]:
best_run = results_df.loc[
    results_df["accuracy"].idxmax()
]

print("Best Run:")
print(best_run)


Best Run:
run_id          8b8aaa3837204aea93a7f863d435fc4e
n_estimators                                  50
max_depth                                      3
accuracy                                0.966667
Name: 0, dtype: object


In [17]:
best_run_id = best_run["run_id"]

model_uri = f"runs:/{best_run_id}/model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="W11_Iris_RandomForest"
)

print("Registered model:", registered_model.name)
print("Model version:", registered_model.version)


Successfully registered model 'W11_Iris_RandomForest'.
2026/09/06 10:00:14 WARNING mlflow.tracking._model_registry.fluent: Run with id 8b8aaa3837204aea93a7f863d435fc4e has no artifacts at artifact path 'model', registering model based on models:/m-a01bfddf45d6438f8703b41cb96a2d16 instead


Registered model: W11_Iris_RandomForest
Model version: 1


Created version '1' of model 'W11_Iris_RandomForest'.


In [18]:
model_version = registered_model.version

loaded_model = mlflow.sklearn.load_model(
    f"models:/W11_Iris_RandomForest/{model_version}"
)

predictions = loaded_model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    predictions
)

print("Loaded model accuracy:", accuracy)


Loaded model accuracy: 0.9666666666666667


Day 3

In [19]:
import joblib

joblib.dump(
    loaded_model,
    "iris_model.pkl"
)

print("Model saved successfully.")


Model saved successfully.


In [20]:
%%writefile app.py

from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)

# Load trained model
model = joblib.load("iris_model.pkl")

@app.route("/")
def home():
    return jsonify({
        "message": "Iris ML Model API is running"
    })


@app.route("/predict", methods=["POST"])
def predict():

    data = request.get_json()

    features = np.array(
        data["features"]
    ).reshape(1, -1)

    prediction = model.predict(features)

    return jsonify({
        "prediction": int(prediction[0])
    })


if __name__ == "__main__":
    app.run(
        host="0.0.0.0",
        port=8000
    )


Writing app.py


In [21]:
!pip install -q flask

In [22]:
!nohup python app.py > api.log 2>&1 &

In [23]:
import requests

sample = {
    "features": [
        5.1,
        3.5,
        1.4,
        0.2
    ]
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=sample
)

print("Status:", response.status_code)
print("Response:", response.json())


Status: 200
Response: {'prediction': 0}


Day 4

In [24]:
!pip install -q ragas datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.2/353.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==

In [25]:
import pandas as pd

data = {
    "question": [
        "What is MLflow?",
        "What is model registry?",
        "What is RAG?",
        "What is experiment tracking?"
    ],

    "answer": [
        "MLflow is a platform for managing the machine learning lifecycle.",
        "A model registry stores and manages different versions of machine learning models.",
        "RAG combines retrieval with generation to provide answers using external knowledge.",
        "Experiment tracking records parameters, metrics, and artifacts from machine learning experiments."
    ],

    "contexts": [
        ["MLflow helps track and manage machine learning experiments."],
        ["A model registry manages model versions and lifecycle stages."],
        ["RAG retrieves relevant documents before generating an answer."],
        ["Experiment tracking records model parameters and evaluation metrics."]
    ],

    "ground_truth": [
        "MLflow is used to manage machine learning experiments and lifecycle.",
        "A model registry manages and versions machine learning models.",
        "RAG retrieves relevant information and uses it to generate answers.",
        "Experiment tracking records parameters, metrics, and artifacts."
    ]
}

rag_df = pd.DataFrame(data)

rag_df


,question,answer,contexts,ground_truth
0,What is MLflow?,MLflow is a platform for managing the machine ...,[MLflow helps track and manage machine learnin...,MLflow is used to manage machine learning expe...
1,What is model registry?,A model registry stores and manages different ...,[A model registry manages model versions and l...,A model registry manages and versions machine ...
2,What is RAG?,RAG combines retrieval with generation to prov...,[RAG retrieves relevant documents before gener...,RAG retrieves relevant information and uses it...
3,What is experiment tracking?,"Experiment tracking records parameters, metric...",[Experiment tracking records model parameters ...,"Experiment tracking records parameters, metric..."


In [26]:
from datasets import Dataset

dataset = Dataset.from_pandas(
    rag_df
)

print(dataset)


Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 4
})


In [1]:
!pip uninstall -y ragas langchain langchain-core langchain-community langchain-google-vertexai -q

!pip install -q \
    "ragas==0.2.15" \
    "langchain==0.3.25" \
    "langchain-core==0.3.59" \
    "langchain-community==0.3.24" \
    "langchain-google-vertexai==2.0.21"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import ragas
import langchain
import langchain_community

print("Ragas:", ragas.__version__)
print("LangChain:", langchain.__version__)
print("LangChain Community:", langchain_community.__version__)

Ragas: 0.2.15
LangChain: 0.3.25
LangChain Community: 0.3.24


In [4]:
import pandas as pd

data = {
    "question": [
        "What is MLflow?",
        "What is model registry?",
        "What is RAG?",
        "What is experiment tracking?"
    ],

    "answer": [
        "MLflow is a platform for managing the machine learning lifecycle.",
        "A model registry stores and manages different versions of machine learning models.",
        "RAG combines retrieval with generation to provide answers using external knowledge.",
        "Experiment tracking records parameters, metrics, and artifacts from machine learning experiments."
    ],

    "contexts": [
        ["MLflow helps track and manage machine learning experiments."],
        ["A model registry manages model versions and lifecycle stages."],
        ["RAG retrieves relevant documents before generating an answer."],
        ["Experiment tracking records model parameters and evaluation metrics."]
    ],

    "ground_truth": [
        "MLflow is used to manage machine learning experiments and lifecycle.",
        "A model registry manages and versions machine learning models.",
        "RAG retrieves relevant information and uses it to generate answers.",
        "Experiment tracking records parameters, metrics, and artifacts."
    ]
}

rag_df = pd.DataFrame(data)

def keyword_score(answer, ground_truth):
    """
    Simple keyword-overlap score.
    Used as a lightweight benchmark for the project.
    """

    answer_words = set(answer.lower().split())
    truth_words = set(ground_truth.lower().split())

    if not truth_words:
        return 0.0

    overlap = answer_words.intersection(truth_words)

    return len(overlap) / len(truth_words)


scores = []

for _, row in rag_df.iterrows():

    score = keyword_score(
        row["answer"],
        row["ground_truth"]
    )

    scores.append(score)

rag_df["answer_score"] = scores

print(rag_df[
    ["question", "answer_score"]
])

                       question  answer_score
0               What is MLflow?      0.500000
1       What is model registry?      1.000000
2                  What is RAG?      0.200000
3  What is experiment tracking?      0.857143


In [5]:
average_score = rag_df["answer_score"].mean()

print(
    f"Average RAG answer score: {average_score:.3f}"
)


Average RAG answer score: 0.639


In [7]:
import mlflow

mlflow.set_experiment("W11_RAG_Evaluation")

with mlflow.start_run(
    run_name="ragas_evaluation"
):

    mlflow.log_metric(
        "average_answer_score",
        float(average_score)
    )

    mlflow.log_param(
        "evaluation_samples",
        len(rag_df)
    )

    rag_df.to_csv(
        "rag_evaluation_results.csv",
        index=False
    )

    mlflow.log_artifact(
        "rag_evaluation_results.csv"
    )

    print("RAG evaluation logged to MLflow.")

2026/09/06 10:09:30 INFO mlflow.tracking.fluent: Experiment with name 'W11_RAG_Evaluation' does not exist. Creating a new experiment.


RAG evaluation logged to MLflow.


Day 5

In [8]:
import mlflow
import pandas as pd

mlflow.set_experiment("W11_Tracked_Evaluated_RAG")

# Knowledge base
knowledge_base = {
    "mlflow": "MLflow is a platform for managing the machine learning lifecycle.",
    "registry": "A model registry manages and versions machine learning models.",
    "rag": "RAG retrieves relevant information and uses it to generate answers.",
    "tracking": "Experiment tracking records parameters, metrics, and artifacts."
}


def retrieve_context(question):
    """
    Retrieve the most relevant knowledge-base entry
    using simple keyword matching.
    """

    question_lower = question.lower()

    for key, context in knowledge_base.items():

        if key in question_lower:
            return context

    return "No relevant information was found."


def generate_answer(question, context):
    """
    Generate an answer using the retrieved context.
    """

    if context == "No relevant information was found.":
        return "I could not find relevant information."

    return context


def run_rag(question):
    """
    Complete RAG pipeline:
    question -> retrieval -> generation
    """

    context = retrieve_context(question)

    answer = generate_answer(
        question,
        context
    )

    return {
        "question": question,
        "context": context,
        "answer": answer
    }


2026/09/06 10:09:31 INFO mlflow.tracking.fluent: Experiment with name 'W11_Tracked_Evaluated_RAG' does not exist. Creating a new experiment.


In [9]:
questions = [
    "What is MLflow?",
    "What is model registry?",
    "What is RAG?",
    "What is experiment tracking?"
]

results = []

for question in questions:

    result = run_rag(question)

    results.append(result)

results_df = pd.DataFrame(results)

results_df


,question,context,answer
0,What is MLflow?,MLflow is a platform for managing the machine ...,MLflow is a platform for managing the machine ...
1,What is model registry?,A model registry manages and versions machine ...,A model registry manages and versions machine ...
2,What is RAG?,RAG retrieves relevant information and uses it...,RAG retrieves relevant information and uses it...
3,What is experiment tracking?,"Experiment tracking records parameters, metric...","Experiment tracking records parameters, metric..."


In [10]:
def evaluate_answer(answer, context):
    """
    Measure whether the generated answer contains
    important information from the retrieved context.
    """

    answer_words = set(answer.lower().split())
    context_words = set(context.lower().split())

    if not context_words:
        return 0.0

    overlap = answer_words.intersection(
        context_words
    )

    return len(overlap) / len(context_words)


results_df["evaluation_score"] = results_df.apply(
    lambda row: evaluate_answer(
        row["answer"],
        row["context"]
    ),
    axis=1
)

results_df


,question,context,answer,evaluation_score
0,What is MLflow?,MLflow is a platform for managing the machine ...,MLflow is a platform for managing the machine ...,1.0
1,What is model registry?,A model registry manages and versions machine ...,A model registry manages and versions machine ...,1.0
2,What is RAG?,RAG retrieves relevant information and uses it...,RAG retrieves relevant information and uses it...,1.0
3,What is experiment tracking?,"Experiment tracking records parameters, metric...","Experiment tracking records parameters, metric...",1.0


In [11]:
average_score = results_df[
    "evaluation_score"
].mean()

with mlflow.start_run(
    run_name="tracked_evaluated_rag_pipeline"
):

    # Parameters
    mlflow.log_param(
        "pipeline_type",
        "Simple RAG"
    )

    mlflow.log_param(
        "number_of_questions",
        len(questions)
    )

    mlflow.log_param(
        "retrieval_method",
        "keyword matching"
    )

    # Evaluation metric
    mlflow.log_metric(
        "average_evaluation_score",
        float(average_score)
    )

    # Save results
    results_df.to_csv(
        "rag_pipeline_results.csv",
        index=False
    )

    mlflow.log_artifact(
        "rag_pipeline_results.csv"
    )

    print(
        "Average evaluation score:",
        round(average_score, 3)
    )

    print(
        "RAG pipeline successfully tracked in MLflow."
    )


Average evaluation score: 1.0
RAG pipeline successfully tracked in MLflow.


In [ ]:
test_question = "What is MLflow?"

result = run_rag(test_question)

print("Question:", result["question"])
print("Context :", result["context"])
print("Answer  :", result["answer"])

assert result["answer"] != ""
assert result["context"] != ""

print("\nPASS: RAG pipeline test")


In [12]:
results_df.to_csv(
    "final_rag_evaluation.csv",
    index=False
)

print("Final evaluation saved successfully.")


Final evaluation saved successfully.
